In [ ]:
# bootstrap: Colab clone + local import of `thinklab` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "00-foundations/rl-and-thinking-models/thinking-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "thinklab").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 03 · Test-time compute for real: pass@k, majority vote, best-of-n, and the price of a correct answer

**Tier:** T1 — sample every problem n times from a real thinking model (`THINKLAB_URL` pointing at
`vllm serve Qwen/Qwen3-0.6B --reasoning-parser qwen3`; 40 problems × (8 + 8 + 16) samples is a few
thousand requests, tens of minutes on a T4 (verify on yours); set `COLLECT = True`). T0 (default) —
the bundled table of the lab's **simulated** model on the same problems (illustrative outcomes,
regenerated by `python -m thinklab.thinking.recorded`).

## The one-minute version

* There are two ways to spend more compute on a question at inference time: think **longer**
  (sequential: more reasoning tokens, or a larger budget) or sample **more** answers and pick one
  (parallel: majority vote, best-of-n with a verifier or reward model). PRIMER §6 "Test-time compute".
* Measure them properly. **pass@k**, the chance that at least one of k samples is right, needs the
  unbiased estimator `1 − C(n−c, k)/C(n, k)` from n ≥ k samples. `1 − (1 − c/n)^k` is biased.
  **pass^k**, the chance that *all* k are right, is the reliability an agent needs, and it *falls*
  with k. **maj@k** needs no verifier, only answers that can be compared.
* pass@k is an upper bound that only a perfect verifier achieves. Majority vote and a noisy reward
  model land below it, and majority vote can *lose* accuracy when a wrong answer is the most
  common one.
* The only fair comparison is **cost per correct answer**: tokens (all samples, all reasoning) ×
  price ÷ accuracy. A thinking token is billed as an output token.

> **Exercise cells** contain `# YOUR CODE HERE` — replace it, then run the **Check** cell below it. A check prints ✅ when it passes. The finished version is in `solutions/`.

In [ ]:
import math, random, statistics
from thinklab import env
from thinklab.report import table
from thinklab.thinking import recorded
from thinklab.thinking.evalset import make_evalset
from thinklab.thinking.ttc import (best_of_n_accuracy, compute_optimal, maj_at_k, majority_vote as ref_vote,
                                   pass_at_k as ref_pass_at_k, pass_hat_k as ref_pass_hat_k)

print(env.describe())
COLLECT = False                                  # T1: set True with THINKLAB_URL pointing at a real server
if COLLECT and env.server_url() and not env.is_simulated(env.server_url(), env.auth_headers()):
    from thinklab.thinking.client import ThinkingClient
    client = ThinkingClient(env.server_url(), headers=env.auth_headers())
    RECORDS = recorded.collect(client, make_evalset(40, seed=0), samples=8, budgets=(256, 1024), budget_samples=8)
    LABEL = f"MEASURED on {client.model}"
else:
    RECORDS = recorded.load()
    LABEL = "SIMULATED model, bundled records (illustrative)"
print(LABEL, "|", len(RECORDS), "records")

def by(mode, budget=None):
    return [r for r in RECORDS if r["mode"] == mode and r["budget"] == budget]

rows = []
for mode, b in (("off", None), ("on", None), ("budget", 128), ("budget", 256), ("budget", 512), ("budget", 1024)):
    sel = by(mode, b)
    if not sel:
        continue
    rows.append({"mode": mode if b is None else f"budget {b}", "problems": len(sel),
                 "samples each": len(sel[0]["correct"]),
                 "accuracy (pass@1)": round(statistics.fmean(c for r in sel for c in r["correct"]), 3),
                 "output tokens": round(statistics.fmean(a + t for r in sel for a, t in zip(r["reasoning_tokens"], r["answer_tokens"])))})
print(table(rows, title=f"[{LABEL}] one sample, by mode"))

## Exercise 3.1 — the unbiased pass@k estimator

Given n samples with c correct, return the probability that a random subset of k of them
contains at least one correct sample: `1 − C(n−c, k) / C(n, k)`. Use the numerically stable
product form from the HumanEval code, `1 − Π_{i=n−c+1..n} (1 − k/i)`. When `n − c < k` every
subset contains a correct sample and the answer is 1.

In [ ]:
def my_pass_at_k(n: int, c: int, k: int) -> float:
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
assert abs(my_pass_at_k(10, 3, 1) - 0.3) < 1e-12 and abs(my_pass_at_k(10, 3, 5) - 0.916667) < 1e-6
assert my_pass_at_k(10, 3, 8) == 1.0 and abs(my_pass_at_k(16, 4, 4) - 0.728022) < 1e-6
assert abs(my_pass_at_k(64, 16, 8) - 0.914746) < 1e-6
assert all(abs(my_pass_at_k(n, c, k) - ref_pass_at_k(n, c, k)) < 1e-12 for n in (5, 8, 16) for c in range(n + 1) for k in range(1, n + 1))
naive = 1 - (1 - 3 / 10) ** 5
print(f"✅ n=10, c=3: pass@5 = {my_pass_at_k(10, 3, 5):.4f}; the naive 1-(1-c/n)^k says {naive:.4f} (biased)")

## Exercise 3.2 — majority vote

Return the most common answer, ignoring `None` (no answer, e.g. cut off while thinking). Break
ties in favour of the answer that appeared *first*. Return `None` if there are no answers.

In [ ]:
def my_vote(answers: list):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
assert my_vote(["4", "5", "4", None, None, None]) == "4"
assert my_vote(["5", "4", "4", "5"]) == "5" and my_vote([None, None]) is None and my_vote([]) is None
rng = random.Random(0)
for _ in range(300):
    xs = [rng.choice(["a", "b", "c", None]) for _ in range(rng.randrange(1, 9))]
    assert my_vote(xs) == ref_vote(xs)
print("✅ majority vote with None ignored and first-seen tie-breaking")

## Worked example: three ways to use k samples

For each problem with 8 thinking-mode samples:

* **pass@k** (perfect verifier, oracle best-of-n), the ceiling;
* **maj@k**, majority vote over k of the 8 samples, averaged over random subsets;
* **best-of-k by a reward model**: the sample with the top score from a *noisy* scorer. In the
  simulated records the scorer gives correct answers +0.8 on average with unit noise; at T1 plug in
  a real reward model.

In [ ]:
for mode in ("off", "on"):
    sel = by(mode)
    out = []
    for k in (1, 2, 4, 8):
        out.append({"k": k,
                    "pass@k": round(statistics.fmean(ref_pass_at_k(len(r["correct"]), sum(r["correct"]), k) for r in sel), 3),
                    "maj@k": round(statistics.fmean(maj_at_k(r["answers"], r["truth"], k) for r in sel), 3),
                    **({"best-of-k (RM)": round(statistics.fmean(best_of_n_accuracy(r["correct"], r["rm_scores"], k) for r in sel), 3)}
                       if sel[0]["rm_scores"] else {}),
                    "pass^k": round(statistics.fmean(ref_pass_hat_k(len(r["correct"]), sum(r["correct"]), k) for r in sel), 3)})
    print(table(out, title=f"[{LABEL}] thinking {mode}"), "\n")

Read across a row. pass@k climbs fastest because it assumes you can recognise the right answer.
maj@k climbs more slowly and saturates at the share of problems where the right answer is the
*most common* one. On problems where a wrong answer dominates, more votes make it *more* certain.
Best-of-k with the noisy reward model rises at first, then can *fall*: the more samples, the more
chances that a wrong one draws the top score. That is reward over-optimisation at inference time,
and the reason to trust a verifier over a reward model where one exists. pass^k falls: the chance
that every one of k attempts succeeds is what an agent running the same step many times sees.
Thinking lifts every column, and each thinking sample costs roughly ten times the tokens.

## Exercise 3.3 — pass^k, the reliability metric

Return the probability that k samples drawn without replacement from n (c correct) are *all*
correct: `C(c, k) / C(n, k)`, as τ-bench computes it.

In [ ]:
def my_pass_hat_k(n: int, c: int, k: int) -> float:
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
assert my_pass_hat_k(10, 3, 5) == 0.0 and abs(my_pass_hat_k(16, 4, 4) - 0.000549) < 1e-6
assert my_pass_hat_k(8, 8, 8) == 1.0 and abs(my_pass_hat_k(10, 9, 2) - 0.8) < 1e-12
print(f"✅ a step that succeeds 90% of the time twice in a row: pass^2 = {my_pass_hat_k(10, 9, 2):.2f}; "
      f"4 of 16: pass^4 = {my_pass_hat_k(16, 4, 4):.6f} while pass@4 = {ref_pass_at_k(16, 4, 4):.3f}")

## Exercise 3.4 — cost per correct answer

Return dollars per correct answer when each question is answered with `samples` samples, each
with `output_tokens` output tokens (reasoning included) at `price_out` dollars per million, and
the chosen answer is right with probability `accuracy`. Then compare three strategies with the
numbers from the table above, at an output price of $2.00 per million tokens (an illustrative
price; the 06 gateway lab's `cost_per_call` has dated ones):
thinking off × maj@8, thinking on × 1 sample, and a 512-token budget × 1 sample.

In [ ]:
def dollars_per_correct(output_tokens: float, accuracy: float, price_out: float, samples: int = 1) -> float:
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
assert dollars_per_correct(1000, 0.5, 2.0) == 0.004 and dollars_per_correct(100, 0.0, 2.0) == math.inf
assert abs(dollars_per_correct(100, 0.4, 2.0, samples=8) - 0.004) < 1e-12
def tok(sel):
    return statistics.fmean(a + t for r in sel for a, t in zip(r["reasoning_tokens"], r["answer_tokens"]))
PRICE = 2.00
strategies = [
    ("thinking off, maj@8", tok(by("off")), statistics.fmean(maj_at_k(r["answers"], r["truth"], 8) for r in by("off")), 8),
    ("thinking on, 1 sample", tok(by("on")), statistics.fmean(c for r in by("on") for c in r["correct"]), 1)]
if by("budget", 512):
    strategies.append(("budget 512, 1 sample", tok(by("budget", 512)),
                       statistics.fmean(c for r in by("budget", 512) for c in r["correct"]), 1))
print(table([{"strategy": s, "tokens/question": round(t * n), "accuracy": round(a, 3),
              "$ per 1K correct": round(1000 * dollars_per_correct(t, a, PRICE, n), 3)} for s, t, a, n in strategies],
            title=f"[{LABEL}] at ${PRICE:.2f} per 1M output tokens"))
print("✅ cost per correct answer = all tokens of all samples x price / accuracy")

## Exercise 3.5 — compute-optimal: the best accuracy for a token budget per question

Build the option list — every (mode, k) with its expected tokens per question (`k` × mean output
tokens of the mode) and its accuracy (maj@k for k > 1, pass@1 for k = 1) — and, for each budget
of tokens per question, pick the most accurate option that fits (cheaper wins ties). This is the
small-scale version of "compute-optimal test-time scaling": the best way to spend a fixed number of
tokens depends on how many there are. With other models and tasks the order can differ; many
short samples sometimes beat one long one. Measure it for yours.

In [ ]:
def options_from_records() -> list:
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
opts = options_from_records()
assert len(opts) >= 8 and all({"option", "tokens", "accuracy"} <= set(o) for o in opts)
picks = [{"budget/question": B, **{k: (round(v, 3) if isinstance(v, float) else v) for k, v in (compute_optimal(opts, B) or {"option": "nothing fits"}).items()}}
         for B in (150, 400, 800, 1500, 3000, 10000)]
print(table(picks, title=f"[{LABEL}] most accurate option within a token budget per question"))
accs = [p.get("accuracy", 0) for p in picks]
assert accs == sorted(accs), "a larger budget can never pick a less accurate option"
print("✅ the best option changes with the budget: direct answers, then budgeted thinking, then full thinking, then votes over thinking samples")

## On a real GPU (T1)

Point `THINKLAB_URL` at a real server, set `COLLECT = True` and re-run. `recorded.collect` asks for
`n=8` choices per request, so vLLM prefills the prompt once and decodes eight sequences in the same
batch. Parallel sampling is cheap in prefill and costs the same KV and decode as eight requests.
There is no reward model in the T1 path. Add one (any sequence-classification reward model served
with vLLM's `--task reward`, verify the flag on your version) or use the verifier column alone.

## In a design review

**Two minutes:** "We can buy accuracy at inference time in two ways: let the model think longer,
or sample several answers and choose. We compare them on cost per *correct* answer, counting every
reasoning token of every sample as output. pass@k is only reachable with a verifier we trust,
which we have for math and code and not for open-ended answers. Majority vote needs no verifier
but saturates, and can amplify a common wrong answer. For agents we report pass^k, because a
workflow that runs a step many times needs all of them to succeed. On our eval, the best option
depends on the token budget per question, so the router, not the model, picks it (notebook 04,
exercise 4.5)."

**Drill 1.** *Why not estimate pass@8 as 1 − (1 − pass@1)^8?* That assumes independent samples
with the same success probability on every problem. The estimator from n ≥ k real samples per
problem is unbiased, and averaging it over problems keeps easy and hard problems separate.

**Drill 2.** *maj@16 is worse than maj@4 on our hardest slice. Bug?* No. When a wrong answer is
the modal answer on a problem, more votes make the wrong answer win more reliably. Check the share
of problems where the right answer is modal.

**Drill 3.** *pass@1 is 90%, so the five-step agent succeeds 90% of the time?* Only if you run each
step once and the steps are independent, which gives 0.9^5 ≈ 59%. If a step must succeed every
time it is retried or repeated, measure pass^k.